## Broken timing C2ST correction 

In [1]:
import importlib,warnings
import numpy as np
import pandas as pd
from time import perf_counter
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

import src.paths as paths
import src.representations as representations
import src.metrics as metrics

importlib.reload(paths)
importlib.reload(representations)
importlib.reload(metrics)

from src.paths import get_project_paths
from src.representations import structural_campaign_summary
from src.metrics import c2st_paired

PATHS=get_project_paths()
DATA_DIR=PATHS["processed"]
RESULTS_DIR=PATHS["results"]

OUT_DIR=RESULTS_DIR/"week8"/"broken_timing_c2st_correction"
OUT_DIR.mkdir(parents=True,exist_ok=True)

BT_REPS=200
BT_N_PAIRS=100
BT_PERMUTATIONS=100
BT_ALPHA=.05
BT_SEVERITIES=[.10,.30,.60]

BT_SAMPLE_BASE_SEED=42
BT_NULL_SAMPLE_BASE_SEED=42000
BT_TEST_BASE_SEED=10000

In [2]:
# load the broken timing frozen artifacts 
campaigns=pd.read_csv(DATA_DIR/"clean_structured_campaigns.csv")
campaigns["campaign_id"]=campaigns["campaign_id"].astype(str)
campaigns["timestamp"]=pd.to_datetime(campaigns["timestamp"])

train_ids=pd.read_csv(
    DATA_DIR/"clean_structural_train_scaled.csv"
)["campaign_id"].astype(str).to_numpy()

eval_ids=pd.read_csv(
    DATA_DIR/"clean_structural_eval_scaled.csv"
)["campaign_id"].astype(str).to_numpy()

files={
    "clean":"broken_timing_structural_clean.csv",
    .10:"broken_timing_structural_10pct.csv",
    .30:"broken_timing_structural_30pct.csv",
    .60:"broken_timing_structural_60pct.csv"
}

bt_raw={}

for key,name in files.items():
    df=pd.read_csv(DATA_DIR/name)
    df["campaign_id"]=df["campaign_id"].astype(str)
    bt_raw[key]=df.set_index("campaign_id").loc[eval_ids].reset_index()

FEATURES=[c for c in bt_raw["clean"].columns if c!="campaign_id"]

print("Features:",len(FEATURES))
for key,df in bt_raw.items():
    print(key,df.shape)

Features: 13
clean (300, 14)
0.1 (300, 14)
0.3 (300, 14)
0.6 (300, 14)


In [3]:
# reconstruct the original frozen scaling 
train_campaigns=campaigns[
    campaigns["campaign_id"].isin(train_ids)
].copy()

train_summary=structural_campaign_summary(train_campaigns)
train_summary["campaign_id"]=train_summary["campaign_id"].astype(str)

train_summary=(
    train_summary
    .set_index("campaign_id")
    .loc[train_ids]
    .reset_index()
)

scaler=StandardScaler()
scaler.fit(train_summary[FEATURES])

X_bt={
    key:scaler.transform(df[FEATURES]).astype(np.float32)
    for key,df in bt_raw.items()
}

for key,X in X_bt.items():
    print(key,X.shape)

clean (300, 13)
0.1 (300, 13)
0.3 (300, 13)
0.6 (300, 13)


In [4]:
# validation 
saved_eval=pd.read_csv(DATA_DIR/"clean_structural_eval_scaled.csv")
saved_eval["campaign_id"]=saved_eval["campaign_id"].astype(str)

saved_eval=(
    saved_eval
    .set_index("campaign_id")
    .loc[eval_ids]
)

max_diff=np.max(
    np.abs(
        X_bt["clean"]
        -saved_eval[FEATURES].to_numpy()
    )
)

print("Max clean scaled difference:",max_diff)

assert np.allclose(
    X_bt["clean"],
    saved_eval[FEATURES].to_numpy(),
    atol=1e-6
)

print("Frozen scaling reproduced.")

Max clean scaled difference: 1.3729358627756483e-07
Frozen scaling reproduced.


In [5]:
# helper for corrected linear c2st
def run_linear_c2st(P,Q,seed):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always",ConvergenceWarning)

        result=c2st_paired(
            P,
            Q,
            test_size=.5,
            n_permutations=BT_PERMUTATIONS,
            seed=seed,
            alpha=BT_ALPHA
        )

    conv_warning=any(
        issubclass(w.category,ConvergenceWarning)
        for w in caught
    )

    return result,conv_warning

200 repetitions

seleccionar 200 clean campaigns independientemente

primeras 100 = $P$

siguientes 100 = $Q$

test seed = 10000 + rep

In [6]:
# recalculate clean null anchor
null_records=[]
start=perf_counter()

for rep in range(BT_REPS):
    rng=np.random.default_rng(
        BT_NULL_SAMPLE_BASE_SEED+rep
    )

    selected=rng.choice(
        len(eval_ids),
        size=2*BT_N_PAIRS,
        replace=False
    )

    P=X_bt["clean"][selected[:BT_N_PAIRS]]
    Q=X_bt["clean"][selected[BT_N_PAIRS:]]

    result,conv_warning=run_linear_c2st(
        P,
        Q,
        BT_TEST_BASE_SEED+rep
    )

    null_records.append({
        "repetition":rep,
        "condition":"clean-null",
        "severity":0.0,
        "accuracy":result["accuracy"],
        "p_value":result["p_value"],
        "reject":result["reject_H0"],
        "convergence_warning":conv_warning
    })

    if (rep+1)%20==0:
        print(f"Null: {rep+1}/{BT_REPS}")

null_runs=pd.DataFrame(null_records)

print("Runtime:",round((perf_counter()-start)/60,2),"minutes")
print("Convergence warnings:",null_runs["convergence_warning"].sum())
print("Type I error:",null_runs["reject"].mean())

assert null_runs["convergence_warning"].sum()==0

Null: 20/200
Null: 40/200
Null: 60/200
Null: 80/200
Null: 100/200
Null: 120/200
Null: 140/200
Null: 160/200
Null: 180/200
Null: 200/200
Runtime: 0.12 minutes
Convergence warnings: 0
Type I error: 0.035


In [7]:
# Recalculate the three Broken Timing power anchors
# reproduce orginial sampling
power_records=[]
start=perf_counter()

for rep in range(BT_REPS):
    rng=np.random.default_rng(
        BT_SAMPLE_BASE_SEED+rep
    )

    idx=rng.choice(
        len(eval_ids),
        size=BT_N_PAIRS,
        replace=False
    )

    P=X_bt["clean"][idx]

    for severity in BT_SEVERITIES:
        Q=X_bt[severity][idx]

        result,conv_warning=run_linear_c2st(
            P,
            Q,
            BT_TEST_BASE_SEED+rep
        )

        power_records.append({
            "repetition":rep,
            "condition":"Broken Timing",
            "severity":severity,
            "n_pairs":BT_N_PAIRS,
            "accuracy":result["accuracy"],
            "p_value":result["p_value"],
            "reject":result["reject_H0"],
            "convergence_warning":conv_warning
        })

    if (rep+1)%20==0:
        elapsed=(perf_counter()-start)/60
        print(f"Power: {rep+1}/{BT_REPS} | {elapsed:.1f} min")

power_runs=pd.DataFrame(power_records)

print("Rows:",len(power_runs))
print("Expected:",BT_REPS*3)
print("Runtime:",round((perf_counter()-start)/60,2),"minutes")
print("Convergence warnings:",power_runs["convergence_warning"].sum())

assert len(power_runs)==600
assert power_runs["convergence_warning"].sum()==0


Power: 20/200 | 0.1 min
Power: 40/200 | 0.1 min
Power: 60/200 | 0.2 min
Power: 80/200 | 0.3 min
Power: 100/200 | 0.3 min
Power: 120/200 | 0.4 min
Power: 140/200 | 0.4 min
Power: 160/200 | 0.6 min
Power: 180/200 | 0.6 min
Power: 200/200 | 0.6 min
Rows: 600
Expected: 600
Runtime: 0.64 minutes
Convergence warnings: 0


In [8]:
# final results
def wilson_interval(k,n,z=1.959963984540054):
    p=k/n
    denom=1+(z**2/n)
    center=(p+z**2/(2*n))/denom
    half=z*np.sqrt((p*(1-p)/n)+(z**2/(4*n**2)))/denom
    return max(0,center-half),min(1,center+half)

summary_rows=[]

# Clean-null
k=int(null_runs["reject"].sum())
lo,hi=wilson_interval(k,len(null_runs))

summary_rows.append({
    "condition":"Clean null",
    "severity":0.0,
    "repetitions":len(null_runs),
    "rejections":k,
    "rate":null_runs["reject"].mean(),
    "ci95_lower":lo,
    "ci95_upper":hi,
    "mean_accuracy":null_runs["accuracy"].mean()
})

# Three defect severities
for severity in BT_SEVERITIES:
    df=power_runs[
        np.isclose(power_runs["severity"],severity)
    ]

    k=int(df["reject"].sum())
    lo,hi=wilson_interval(k,len(df))

    summary_rows.append({
        "condition":"Broken Timing",
        "severity":severity,
        "repetitions":len(df),
        "rejections":k,
        "rate":df["reject"].mean(),
        "ci95_lower":lo,
        "ci95_upper":hi,
        "mean_accuracy":df["accuracy"].mean()
    })

corrected_summary=pd.DataFrame(summary_rows)

display(corrected_summary)

,condition,severity,repetitions,rejections,rate,ci95_lower,ci95_upper,mean_accuracy
0,Clean null,0.0,200,7,0.035,0.017056,0.070471,0.49935
1,Broken Timing,0.1,200,5,0.025,0.010725,0.057178,0.51790
2,Broken Timing,0.3,200,167,0.835,0.777342,0.880031,0.56990
3,Broken Timing,0.6,200,200,1.000,0.981155,1.000000,0.66095


In [9]:
# save without overwritting
null_runs.to_csv(
    OUT_DIR/"c2st_linear_broken_timing_n100_null_runs.csv",
    index=False
)

power_runs.to_csv(
    OUT_DIR/"c2st_linear_broken_timing_n100_power_runs.csv",
    index=False
)

corrected_summary.to_csv(
    OUT_DIR/"c2st_linear_broken_timing_n100_corrected_summary.csv",
    index=False
)

print("Saved to:",OUT_DIR)

Saved to: /Users/jessica/fidelity-metric-lab/results/week8/broken_timing_c2st_correction


The four corrected results are exactly the same as the ones you had before:

This is methodologically sound. The warning was genuine and needed addressing, but after allowing L-BFGS to converge with `max_iter=5000`, the scientific conclusions and pruning decisions remain unchanged.
It is also confirmed that the calibration remains valid:
$$
\text{Type I}=0.035,\qquad 95\%\,CI=[0.0171,\;0.0705].
$$
And for Broken Timing:

$$
10\%:\;0.025
$$
$$
30\%:\;0.835
$$
$$
60\%:\;1.000
$$
Therefore, the pruning rule yields exactly the same results for C2ST-Linear / Broken Timing:
- 10%: .025 falls within the null range → prune $n=50$, keep $n=200$.
- 30%: .835 represents a transition → keep both $n=50$ and $n=200$.
- 60%: power = 1.00 → prune $n=200$, keep $n=50$.
You do not need to re-run the other four C2ST-Linear scenarios.